In [1]:
import torch
from torch import nn
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from model.decoder_block import Decoder_Block

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size: int, context_length: int, model_dim: int, num_blocks: int, num_heads: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, model_dim)
        self.pos_embedding = nn.Embedding(context_length, model_dim)
        self.transformer_blocks = nn.Sequential()
        for _ in range(num_blocks):
            self.transformer_blocks.append(Decoder_Block(model_dim, num_heads))
        self.layer_norm_three = nn.LayerNorm(model_dim)
        self.vocab_projection = nn.Linear(model_dim, vocab_size)

    def forward(self, context):
        embedded = self.token_embedding(context)
        context_length = context.shape[1]
        positions = torch.arange(context_length).to(device)
        embedded = embedded + self.pos_embedding(positions)

        raw_output = self.vocab_projection(self.layer_norm_three(self.transformer_blocks(embedded)))
        # raw_output is BxTxV, where V is the vocabulary size
        return raw_output

In [5]:
vocab_size = 5 # THe number of different tokens the model recognizes
context_length = 5 # How many tokens back the model can read
model_dim = 16 # Feature dimensionality for embeddings and attention
num_blocks = 4 # Number of repetitions of Transformer block
num_heads = 4 # Number of self attention instances

gpt = Decoder(vocab_size, context_length, model_dim, num_blocks, num_heads, mask=True)
gpt = gpt.to(device)

context = [['With', 'great', 'power', 'comes', 'great']] # BxT
mapping = {'with': 0, 'great': 1, 'power': 2, 'comes': 3, 'responsibility': 4}
context_indices = [[mapping[token.lower()] for token in seq] for seq in context]


context = torch.tensor(context_indices, dtype=torch.long).to(device)

probabilities = gpt(context)
print(probabilities)

tensor([[[-0.1428,  0.2789, -0.4691,  0.5855, -0.4161],
         [ 0.5853, -0.4109,  0.1774, -0.0384, -1.0528],
         [ 0.7083, -0.6464, -0.7246, -0.0545, -0.5213],
         [ 0.5258,  0.9025,  0.5950, -0.5040,  0.4112],
         [ 0.5523,  0.7344,  0.7147, -0.1419,  0.3880]]], device='cuda:0',
       grad_fn=<ViewBackward0>)
